# Assignment 2: Convolutions - Stride Comparison

**Based on:** [Generative Deep Learning 2nd Edition](https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition) - `convolutions.ipynb`

This notebook builds two versions of a CNN model:
1. **Original version**: Uses stride=2 (as in the textbook)
2. **Custom stride version**: Uses stride=3 (different from 2)

We train both on MNIST and compare accuracy and output characteristics.

## Setup and Imports

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from skimage import data
from skimage.color import rgb2gray
from skimage.transform import resize
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

## Part 1: Textbook Convolution Code (Visualization)

Reproduced from the textbook to understand how convolutions and stride affect output.

### Original Input Image

In [ ]:
im = rgb2gray(data.coffee())
im = resize(im, (64, 64))
print("Image shape:", im.shape)

plt.figure(figsize=(4, 4))
plt.axis("off")
plt.imshow(im, cmap="gray")
plt.title("Original Image (64x64)")
plt.show()

### Horizontal Edge Filter (Stride 1 - Full Resolution)

In [ ]:
filter1 = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])

new_image = np.zeros(im.shape)
im_pad = np.pad(im, 1, "constant")

for i in range(im.shape[0]):
    for j in range(im.shape[1]):
        try:
            new_image[i, j] = (
                im_pad[i - 1, j - 1] * filter1[0, 0]
                + im_pad[i - 1, j] * filter1[0, 1]
                + im_pad[i - 1, j + 1] * filter1[0, 2]
                + im_pad[i, j - 1] * filter1[1, 0]
                + im_pad[i, j] * filter1[1, 1]
                + im_pad[i, j + 1] * filter1[1, 2]
                + im_pad[i + 1, j - 1] * filter1[2, 0]
                + im_pad[i + 1, j] * filter1[2, 1]
                + im_pad[i + 1, j + 1] * filter1[2, 2]
            )
        except:
            pass

print("Output shape (stride=1):", new_image.shape)
plt.figure(figsize=(4, 4))
plt.axis("off")
plt.imshow(new_image, cmap="Greys")
plt.title("Horizontal Edge Filter - Stride 1")
plt.show()

### Horizontal Edge Filter with Stride 2 (Textbook Original)

In [ ]:
filter1 = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])
stride = 2

new_image = np.zeros((int(im.shape[0] / stride), int(im.shape[1] / stride)))
im_pad = np.pad(im, 1, "constant")

for i in range(0, im.shape[0], stride):
    for j in range(0, im.shape[1], stride):
        try:
            new_image[int(i / stride), int(j / stride)] = (
                im_pad[i - 1, j - 1] * filter1[0, 0]
                + im_pad[i - 1, j] * filter1[0, 1]
                + im_pad[i - 1, j + 1] * filter1[0, 2]
                + im_pad[i, j - 1] * filter1[1, 0]
                + im_pad[i, j] * filter1[1, 1]
                + im_pad[i, j + 1] * filter1[1, 2]
                + im_pad[i + 1, j - 1] * filter1[2, 0]
                + im_pad[i + 1, j] * filter1[2, 1]
                + im_pad[i + 1, j + 1] * filter1[2, 2]
            )
        except:
            pass

print("Output shape (stride=2):", new_image.shape)
plt.figure(figsize=(4, 4))
plt.axis("off")
plt.imshow(new_image, cmap="Greys")
plt.title("Horizontal Edge Filter - Stride 2 (Original)")
plt.show()

### Horizontal Edge Filter with Stride 3 (Custom)

In [ ]:
filter1 = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]])
stride = 3  # Custom stride (different from 2)

# Output size: ceil(input/stride) to match iteration count
out_h = (im.shape[0] + stride - 1) // stride
out_w = (im.shape[1] + stride - 1) // stride
new_image = np.zeros((out_h, out_w))
im_pad = np.pad(im, 1, "constant")

for oi, i in enumerate(range(0, im.shape[0], stride)):
    for oj, j in enumerate(range(0, im.shape[1], stride)):
        try:
            if oi < out_h and oj < out_w:
                new_image[oi, oj] = (
                    im_pad[i - 1, j - 1] * filter1[0, 0]
                    + im_pad[i - 1, j] * filter1[0, 1]
                    + im_pad[i - 1, j + 1] * filter1[0, 2]
                    + im_pad[i, j - 1] * filter1[1, 0]
                    + im_pad[i, j] * filter1[1, 1]
                    + im_pad[i, j + 1] * filter1[1, 2]
                    + im_pad[i + 1, j - 1] * filter1[2, 0]
                    + im_pad[i + 1, j] * filter1[2, 1]
                    + im_pad[i + 1, j + 1] * filter1[2, 2]
                )
        except:
            pass

print("Output shape (stride=3):", new_image.shape)
plt.figure(figsize=(4, 4))
plt.axis("off")
plt.imshow(new_image, cmap="Greys")
plt.title("Horizontal Edge Filter - Stride 3 (Custom)")
plt.show()

## Part 2: CNN Models for Classification

We build two CNN models that mirror the stride behavior from the textbook:
- **Model 1 (Original)**: Stride=2 in convolutional layers
- **Model 2 (Custom)**: Stride=3 in convolutional layers

In [ ]:
# Load MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize and add channel dimension
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

### Model 1: Original (Stride=2)

In [1]:
def build_model_stride2():
    """CNN with stride=2 (original textbook style)."""
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, strides=2, activation="relu", padding="same"),
        layers.Conv2D(64, 3, strides=2, activation="relu", padding="same"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ])
    return model

model_stride2 = build_model_stride2()
model_stride2.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model_stride2.summary()

NameError: name 'keras' is not defined

### Model 2: Custom Stride (Stride=3)

In [ ]:
def build_model_stride3():
    """CNN with stride=3 (custom, different from textbook)."""
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, strides=3, activation="relu", padding="same"),
        layers.Conv2D(64, 3, strides=3, activation="relu", padding="same"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(10, activation="softmax")
    ])
    return model

model_stride3 = build_model_stride3()
model_stride3.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model_stride3.summary()

## Part 3: Training Both Models

In [ ]:
EPOCHS = 5
BATCH_SIZE = 128

print("=" * 50)
print("Training Model 1 (Stride=2 - Original)")
print("=" * 50)
history_stride2 = model_stride2.fit(
    x_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    verbose=1
)

In [ ]:
print("=" * 50)
print("Training Model 2 (Stride=3 - Custom)")
print("=" * 50)
history_stride3 = model_stride3.fit(
    x_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    verbose=1
)

## Part 4: Evaluation

In [ ]:
print("\n" + "=" * 50)
print("EVALUATION ON TEST SET")
print("=" * 50)

loss2, acc2 = model_stride2.evaluate(x_test, y_test, verbose=0)
loss3, acc3 = model_stride3.evaluate(x_test, y_test, verbose=0)

print(f"\nModel 1 (Stride=2): Test Loss = {loss2:.4f}, Test Accuracy = {acc2:.4f}")
print(f"Model 2 (Stride=3): Test Loss = {loss3:.4f}, Test Accuracy = {acc3:.4f}")
print(f"\nAccuracy difference: {acc2 - acc3:+.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_stride2.history["accuracy"], label="Stride=2 Train")
axes[0].plot(history_stride2.history["val_accuracy"], label="Stride=2 Val")
axes[0].plot(history_stride3.history["accuracy"], label="Stride=3 Train")
axes[0].plot(history_stride3.history["val_accuracy"], label="Stride=3 Val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].set_title("Accuracy Comparison")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history_stride2.history["loss"], label="Stride=2 Train")
axes[1].plot(history_stride2.history["val_loss"], label="Stride=2 Val")
axes[1].plot(history_stride3.history["loss"], label="Stride=3 Train")
axes[1].plot(history_stride3.history["val_loss"], label="Stride=3 Val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_title("Loss Comparison")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Sample predictions comparison
n_samples = 8
pred2 = model_stride2.predict(x_test[:n_samples])
pred3 = model_stride3.predict(x_test[:n_samples])

fig, axes = plt.subplots(3, n_samples, figsize=(14, 5))
for i in range(n_samples):
    axes[0, i].imshow(x_test[i].squeeze(), cmap="gray")
    axes[0, i].set_title(f"True: {y_test[i]}")
    axes[0, i].axis("off")
    axes[1, i].bar(range(10), pred2[i], color="steelblue", alpha=0.8)
    axes[1, i].set_xticks(range(10))
    axes[1, i].set_title(f"S2 pred: {pred2[i].argmax()}")
    axes[2, i].bar(range(10), pred3[i], color="coral", alpha=0.8)
    axes[2, i].set_xticks(range(10))
    axes[2, i].set_title(f"S3 pred: {pred3[i].argmax()}")
plt.suptitle("Sample Predictions: Stride=2 vs Stride=3")
plt.tight_layout()
plt.show()

## Part 5: Interpretation for Submission

### Observations about Output and Accuracy

**1. Output Size and Spatial Resolution**
- **Stride=1**: Produces output the same size as input (64×64). Every pixel is convolved; maximum spatial detail preserved.
- **Stride=2**: Output is half the size (32×32). Downsampling by 2×; some spatial information is skipped.
- **Stride=3**: Output is ~1/3 the size (22×22 for 64×64 input). More aggressive downsampling; fewer output positions.

**2. Effect on CNN Classification**
- **Stride=2 (Original)**: Common choice in CNNs. Balances computation (fewer feature map positions) with enough spatial resolution for MNIST digits. Typically achieves **higher accuracy** because it retains more useful spatial structure.
- **Stride=3 (Custom)**: More aggressive downsampling. Fewer parameters in later layers, faster training, but **lower accuracy** because:
  - Critical fine-grained features (e.g., stroke curvature in digits) may be lost
  - Feature maps become very small quickly (28→10→4 with two stride-3 layers), limiting discriminative capacity

**3. Accuracy Interpretation**
- Expect **Stride=2** to outperform **Stride=3** on MNIST.
- Stride=2 is a standard design choice; stride=3 is rarely used for small images like 28×28.
- For larger images (e.g., 224×224), stride=3 might be acceptable in early layers.

**4. Trade-offs**
- **Larger stride** → smaller feature maps → less computation, fewer parameters, faster training, but potential loss of accuracy.
- **Smaller stride** → larger feature maps → more computation, more capacity, often better accuracy for fine-grained tasks.